In [1]:
import numpy as np
import json
from classy import Class
import matplotlib.pyplot as plt
from scipy.interpolate import interp1d
import cosmoprimo
import os

from FishLSS.fisherForecast import fisherForecast
from FishLSS.experiment import experiment

In [2]:
# filename for example survey
bfn = 'DESI3'
bd = '/home/adrien/PDM/code/PDM2026_wsl/derivatives/'

log_path = '/home/adrien/miniconda3/envs/fishlss/lib/python3.11/site-packages/FishLSS/bao_recon/log.npy'

In [3]:
f = open('../derivatives/output/'+bfn+'/summary.json')
summary = json.load(f)

In [4]:
print('summary = ')
for keys in summary.keys():
    print(keys, summary[keys])

summary = 
Forecast name DESI3
Edges of redshift bins [0.15, 0.25, 0.35, 0.45000000000000007, 0.55, 0.65, 0.7500000000000001, 0.8500000000000001, 0.9500000000000001, 1.05, 1.15, 1.25, 1.35, 1.45, 1.55, 1.65, 1.75, 1.85, 1.95, 2.0500000000000003, 2.15, 2.25, 2.35, 2.45, 2.5500000000000003, 2.65, 2.75, 2.85, 2.95]
Centers of redshift bins [0.2, 0.3, 0.4, 0.5, 0.6000000000000001, 0.7000000000000001, 0.8, 0.9000000000000001, 1.0, 1.1, 1.2, 1.3, 1.4, 1.5, 1.6, 1.7, 1.8, 1.9, 2.0, 2.1, 2.2, 2.3, 2.4000000000000004, 2.5, 2.6, 2.7, 2.8, 2.9000000000000004]
Linear Eulerian bias in each bin [1.65, 1.65, 1.825, 2.0, 2.0, 2.0, 2.0, 2.0, 1.6, 1.2, 1.2, 1.2, 1.2, 1.2, 1.2, 1.6, 2.0, 2.0, 2.0, 2.0, 2.0, 2.0, 2.0, 2.0, 2.0, 2.0, 2.0, 2.0]
Number density in each bin [0.0008, 0.00075, 0.0005499999999999999, 0.0004, 0.0004, 0.0004, 0.00034999999999999994, 0.0003, 0.0003, 0.0003, 0.0003, 0.00025, 0.0002, 0.000175, 0.00012499999999999995, 0.000225, 0.00035, 0.00018500000000000038, 2e-05, 2e-05, 2e-05, 1.75

In [5]:
params = summary['CLASS default parameters']
cosmo = Class() 
cosmo.set(params) 
cosmo.compute() 

In [6]:
# load fiducial linear bias/number density from table
zs,bs,ns = np.genfromtxt('/home/adrien/PDM/code/PDM2026_wsl/FishLSS_script/' + bfn + '.txt').T

# assume zs spans the full survey
ze = summary['Edges of redshift bins']
zmin = ze[0]
zmax = ze[-1]

z_centers = (np.array(ze[1:])+np.array(ze[:-1]))/2

# interpolate
b = interp1d(zs,bs)
n = interp1d(zs,ns)

nbins = len(ze)-1
fsky = summary['fsky']

exp = experiment(zmin=zmin, zmax=zmax, nbins=nbins, fsky=fsky, b=b, n=n)

name = summary['Forecast name']
forecast = fisherForecast(experiment=exp,cosmo=cosmo,name=name,basedir=bd)

In [7]:
basis = np.array(['alpha_perp','alpha_parallel','b'])

# set recon = True, so that we perform BAO reconstruction when computing the power spectrum
forecast.recon = True

# set the "marginalized parameters", aka the derivatives, to be [alpha's, linear b]
forecast.free_params = basis

derivs = forecast.load_derivatives(basis) # load the pre computed derivatives

In [8]:
F = lambda i: forecast.gen_fisher(basis, 100, derivatives=derivs, zbins=np.array([i]))
Fs = [F(i) for i in range(nbins)]
Fs = np.array(Fs)

In [9]:
Finvs = [np.linalg.inv(Fs[i]) for i in range(nbins)]
saperp = [np.sqrt(Finvs[i][0,0]) for i in range(nbins)]
saparr = [np.sqrt(Finvs[i][1,1]) for i in range(nbins)]

### Save $\alpha_\perp$ $\alpha_\parallel$ errors

In [10]:
def create_DESI_fid_data(redshifts, DESI_style=False):
    cosmo_planck = cosmoprimo.fiducial.DESI()
    bkg = cosmo_planck.get_background(engine="class")
    thermo = cosmo_planck.get_thermodynamics()
    rdrag = thermo.rs_drag  # no parentheses needed, it's a property

    DM = bkg.comoving_angular_distance(redshifts)
    DH = 1 / bkg.efunc(redshifts) * 2997.92  # c/H(z) in Mpc, c=299792 km/s, H0 in km/s/Mpc
    DV = (redshifts * DM**2 * DH)**(1/3)
    DM_rd = DM / rdrag
    DH_rd = DH / rdrag
    DV_rd = DV / rdrag

    data_typ = ['DH_over_rs', 'DM_over_rs', 'DV_over_rs']

    fake_data = []
    if DESI_style:
        for i in range(1, len(redshifts)):
            line = [f"{redshifts[i]:.8e}", f"{DM_rd[i]:.8e}", data_typ[1]]
            fake_data.append(line)
            line = [f"{redshifts[i]:.8e}", f"{DH_rd[i]:.8e}", data_typ[0]]
            fake_data.append(line)

        line = [f"{float(redshifts[0]):.8e}", f"{DV_rd[0]:.8e}", data_typ[2]]
        fake_data.insert(0, line)
    else:
        for i in range(len(redshifts)):
            line = [f"{redshifts[i]:.8e}", f"{DM_rd[i]:.8e}", data_typ[1]]
            fake_data.append(line)
            line = [f"{redshifts[i]:.8e}", f"{DH_rd[i]:.8e}", data_typ[0]]
            fake_data.append(line)

    return np.array(fake_data)

def save_mean_data(fake_data, folder, filename, overwrite=False):
    if not os.path.exists(folder):
        os.makedirs(folder)
    f = folder + '/' + filename
    if os.path.exists(f):
        if overwrite:
            print("Overwriting file:", f)
            np.savetxt(f, fake_data, fmt="%s")
        else:
            print("File already exists:", f)
    else:
        np.savetxt(f, fake_data, fmt="%s")

def cov_from_Fisher_with_units(redshifts, Fish_inv, nbins, with_units=True, DESI_style=False):
    cov_mat = np.zeros((2*nbins, 2*nbins))
    mean = create_DESI_fid_data(redshifts, DESI_style=False)

    if not with_units:
        mean[:,1] = 1.0


    for i in range(nbins):
        if DESI_style:
            if i == 0:
                sig2_DM = Fish_inv[i][0,0] * float(mean[0,1])**2
                sig2_DH = Fish_inv[i][1,1] * float(mean[1,1])**2
                DV = (redshifts[i] * float(mean[0,1])**2 * float(mean[1,1]))**(1/3)
                sig2_DV = DV**2 * ( (2/3)**2 * (sig2_DM / float(mean[0,1])**2) + (1/3)**2 * (sig2_DH / float(mean[1,1])**2) )
                cov_mat = np.zeros((2*nbins-1, 2*nbins-1))
                cov_mat[0,0] = sig2_DV
            else:
                cov_mat[2*i-1,2*i-1] = Fish_inv[i][0,0] * float(mean[2*i,1])**2
                cov_mat[2*i,2*i] = Fish_inv[i][1,1] * float(mean[2*i+1,1])**2
                cov_mat[2*i-1,2*i] = Fish_inv[i][0,1] * float(mean[2*i,1]) * float(mean[2*i+1,1])
                cov_mat[2*i,2*i-1] = Fish_inv[i][1,0] * float(mean[2*i+1,1]) * float(mean[2*i,1])

        else:
            cov_mat[2*i,2*i] = Fish_inv[i][0,0] * float(mean[2*i,1])**2
            cov_mat[2*i+1,2*i+1] = Fish_inv[i][1,1] * float(mean[2*i+1,1])**2
            cov_mat[2*i,2*i+1] = Fish_inv[i][0,1] * float(mean[2*i,1]) * float(mean[2*i+1,1])
            cov_mat[2*i+1,2*i] = Fish_inv[i][1,0] * float(mean[2*i+1,1]) * float(mean[2*i,1])
    
    return cov_mat


def save_a_cov_mat(cov_mat, folder, filename, overwrite=False):

    if not os.path.exists(folder):
        os.makedirs(folder)
    f = folder + '/' + filename
    if os.path.exists(f):
        if overwrite:
            print("Overwriting file:", f)
            np.savetxt(f, cov_mat, fmt="%.8e")
        else:
            print("File already exists:", f)
    else:
        np.savetxt(f, cov_mat, fmt="%.8e")

In [11]:
overwrite = False

In [12]:
folder_to_save = '/home/adrien/PDM/code/PDM2026_wsl/cov_mat/' + bfn
if not os.path.exists(folder_to_save):
    os.makedirs(folder_to_save)

save_a_cov_mat(cov_from_Fisher_with_units(z_centers, Finvs, nbins, with_units=False, DESI_style=False),
                folder_to_save , '/cov_alpha.txt', overwrite=overwrite)
save_a_cov_mat(cov_from_Fisher_with_units(z_centers, Finvs, nbins, DESI_style=False), 
                folder_to_save , '/cov_DM_DH.txt', overwrite=overwrite)

f = folder_to_save + '/redshifts.txt'
if overwrite:
    print("Overwriting file:", f)
    np.savetxt(f, z_centers, fmt="%.8e")
else:
    if not os.path.exists(f):
        np.savetxt(f, z_centers, fmt="%.8e")
    else:
        print("File already exists:", f)